# 03. Multi-output LightGBM 학습 및 예측 (Target-Centric)

## 📋 개요
이 노트북은 **타깃 중심 정렬(Target-Centric Alignment)** 방식을 사용하여 모델을 학습합니다.
- **핵심 논리**: $t$ 시점의 가격을 맞추기 위해 $t-i$ 시점의 데이터를 참조하는 Direct Forecasting 전략을 사용합니다.
- **데이터 정렬**: 예측 결과의 날짜(`date`)가 실제 예측 대상일과 일치하도록 구성하여 분석의 직관성을 높였습니다.
- **하이브리드 구조**: 5일치(1주일) Chunk를 한 번에 예측하는 Multi-output 모델을 구축합니다.

## 🔧 Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import copy
from tqdm import tqdm

from src.utils.config import load_config
from src.models.lightgbm_model import LightGBMModel
from src.modeling.trainer import WalkForwardTrainer
from src.models.artifact import save_model_artifact

warnings.filterwarnings('ignore')

In [ ]:
# 1. 설정 로드
cfg = load_config()
ref_date = cfg['project']['reference_date']
train_cfg = cfg['training']

# 2. 경로 설정
input_dir = Path(cfg['paths']['processed_dir']) / ref_date
result_base_dir = Path(cfg['paths']['result_dir']) / ref_date
model_dir = Path(cfg['paths']['model_dir']) / ref_date
csv_pred_dir = result_base_dir / 'csv'

model_dir.mkdir(parents=True, exist_ok=True)
csv_pred_dir.mkdir(parents=True, exist_ok=True)

print(f"🚀 [Step 3] Multi-output 학습 시작 (기준일: {ref_date})")

## 1️⃣ 데이터 로드
02단계에서 생성된 통합 Feature 데이터셋을 로드합니다. 개별 시점의 피처 시프트는 Trainer 내부에서 타깃별로 수행됩니다.

In [ ]:
print("📥 Loading dataset...")
df = pd.read_parquet(input_dir / "dataset.parquet")
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

feature_cols = [c for c in df.columns if c.startswith('feature_') or c in ['liquidity_score', 'risk_composite']]

print(f"   - 학습 데이터 행수: {len(df):,}")
print(f"   - 사용 피처 수: {len(feature_cols)}")

## 2️⃣ 모델 및 Trainer 초기화
Multi-output을 지원하는 `LightGBMModel`을 생성하고, 타깃 중심 정렬 로직이 포함된 `WalkForwardTrainer`를 설정합니다.

In [ ]:
print("🔧 Initializing Multi-output Model & Trainer...")

model = LightGBMModel(
    model_version=f"v1_multi_{ref_date}",
    params=train_cfg['lgbm_params'],
    feature_list=feature_cols,
    categorical_features=[] # Ticker 제외 통합 모델
)

horizons = train_cfg.get('horizons', [1, 2, 3, 4, 5])
target_base = train_cfg.get('target_col_name', 'target_log_close')

# 모델의 타깃 식별자 설정
model.target_columns = [f"{target_base}_h{h}" for h in horizons]

trainer = WalkForwardTrainer(
    model=model,
    feature_cols=feature_cols,
    target_col_name=target_base, # 공통 타깃명 전달
    horizons=horizons,           # 시차 리스트 전달
    date_col='date'
)

## 3️⃣ Walk-Forward 학습 실행
각 시점($t$)의 가격을 정답으로 두고, $t-1, t-2, \dots, t-5$의 피처를 각각 매칭하여 5개의 내부 모델을 학습합니다.

In [ ]:
print("🏃 Running Target-Centric Walk-Forward Training...")

results = trainer.run(
    df=df,
    train_end=train_cfg['train_end'],
    valid_window_days=train_cfg['valid_window_days'],
    test_window_days=train_cfg['test_window_days'],
    num_valid=train_cfg['num_valid'],
    fit_kwargs={
        'num_boost_round': 1000,
        'callbacks': [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0)
        ]
    }
)

## 4️⃣ 결과 저장 및 성능 요약

In [ ]:
print("\n💾 Saving predictions & model artifact...")

pred_df = results['test_predictions']

# 로그 예측값을 실제 가격으로 변환
for col in model.target_columns:
    if f'pred_{col}' in pred_df.columns:
        pred_df[f'price_pred_{col}'] = np.exp(pred_df[f'pred_{col}'])

# A. 통합 Parquet 저장 (파이프라인용)
full_parquet_path = result_base_dir / "predictions.parquet"
pred_df.to_parquet(full_parquet_path, index=False)
print(f"   - [Total] Saved Parquet: {full_parquet_path}")

# B. 종목별 개별 CSV 저장 (디버깅/사람용)
print(f"   - [Individual] Saving ticker CSVs to {csv_pred_dir}...")

# ticker_name_map 로드 (01단계 master 활용)
try:
    master_path = Path(cfg['paths']['raw_dir']) / ref_date / f"ticker_master_{ref_date}.csv"
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
except Exception:
    ticker_name_map = {}

for ticker, group in tqdm(pred_df.groupby('ticker'), desc="Saving CSVs"):
    name = ticker_name_map.get(str(ticker), f"ticker_{ticker}")
    safe_name = str(name).replace('/', '_').replace('\\', '_')
    group.to_csv(csv_pred_dir / f"{safe_name}.csv", index=False, encoding='utf-8-sig')

# C. 모델 아티팩트 저장
save_model_artifact(
    model_name="lightgbm_multi",
    model_version=f"v1_{ref_date}",
    model_object=results['final_model'],
    metadata={
        "test_metrics": results['test_metrics'],
        "target_columns": model.target_columns
    },
    model_dir=model_dir
)

print("\n✅ [Step 3] 모든 산출물 저장 완료")
display(pred_df.head())

## 🏁 모델 학습 완료
- **예측 데이터**: `predictions.parquet`에서 각 날짜별로 $t-1 \sim t-5$ 시점에 예측한 값들을 확인할 수 있습니다.
- **다음 단계**: 생성된 Chunk 단위 예측값을 기반으로 60일까지의 **재귀적 확장(Recursive Extension)** 및 전략 백테스트를 수행하십시오.